# Desproporcionalidade

In [1]:
import sys
import pandas as pd
sys.path.append('../../src/')

# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Banco em arquivo (persiste durante a sessão).
#%sql duckdb:///content/vigimed.duckdb
# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

# 1. Iniciando a seção

In [2]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


# 2. Criando as tabelas

In [12]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_atc\dim_atc.parquet');


DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_medicamentos\fat_medicamentos.parquet');

DROP TABLE IF EXISTS dim_notificacoes;
CREATE TABLE dim_notificacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_notificacoes\dim_notificacoes.parquet');

DROP TABLE IF EXISTS fat_reacoes;
CREATE TABLE fat_reacoes AS
SELECT IDENTIFICACAO_NOTIFICACAO as id, * 
FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_reacoes\fat_reacoes.parquet');
 

Running query in 'duckdb:///:memory:'

,Success


# 3. Checando as colunas das tabelas

In [13]:
%%sql 
select * from dim_notificacoes
LIMIT 5

Running query in 'duckdb:///:memory:'

,id,PK_NOTIFICACAO,UF,UF_VALOR,UF_DESCRICAO,TIPO_ENTRADA_VIGIMED,TIPO_ENTRADA_VIGIMED_VALOR,RECEBIDO_DE,RECEBIDO_DE_VALOR,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,TIPO_NOTIFICACAO_VALOR,NOTIFICACAO_PARENT_CHILD,NOTIFICACAO_PARENT_CHILD_VALOR,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,GRUPO_IDADE,GRUPO_IDADE_VALOR,GRUPO_IDADE_DESCRICAO,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_UNIDADE,...,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVE_VALOR,GRAVIDADE,GRAVIDADE_VALOR,DESFECHO,DESFECHO_VALOR,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO_VALOR,DURACAO_UNIDADE,RELACAO_MEDICAMENTO_EVENTO,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,ACAO_ADOTADA_VALOR,NOTIFICADOR,NOTIFICADOR_VALOR,HASH,VALIDADE_REGISTRO_INICIAL,VALIDADE_REGISTRO_FINAL,ATUALIZADO_EM,FLAG_ATUAL
0,BR-ANVISA-300212656,1,25,SP,SAO PAULO,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,BR-ANVISA-300212656,2023-09-28,2023-09-28,NaT,1,NOTIFICACAO ESPONTANEA,0,NAO INFORMADO,1990-01-31,30,ANO(S),0,DESCONHECIDO,DESCONHECIDO,0,DESCONHECIDO,...,68.0,165,HEMIPARESIA,1,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,1,RECUPERADO,2021-01-25,2021-02-06,12,DIA(S),2,CONCOMITANTE,TAMISA,3,NAO APLICAVEL,4,MEDICO,35cb7fb032989ed964911f71a7f71e660476984c53030f...,2025-11-17,9999-12-31,2025-11-17,True
1,BR-ANVISA-300208322,2,25,SP,SAO PAULO,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,BR-ANVISA-300208322,2023-09-01,2023-09-01,NaT,1,NOTIFICACAO ESPONTANEA,0,NAO INFORMADO,NaT,0,DESCONHECIDO,0,DESCONHECIDO,DESCONHECIDO,0,DESCONHECIDO,...,DESCONHECIDO,DESCONHECIDO,CEFALEIA,2,NAO,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,2,DESCONHECIDO,2021-01-22,NaT,0,DESCONHECIDO,1,SUSPEITO,CORONAVAC,3,NAO APLICAVEL,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,bef737a8ddcded5ede938b991a19f16b6546081b16df82...,2025-11-17,9999-12-31,2025-11-17,True
2,BR-ANVISA-300214015,3,25,SP,SAO PAULO,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,BR-ANVISA-300214015,2023-10-06,2023-10-06,NaT,1,NOTIFICACAO ESPONTANEA,0,NAO INFORMADO,1971-05-22,49,ANO(S),0,DESCONHECIDO,DESCONHECIDO,0,DESCONHECIDO,...,DESCONHECIDO,DESCONHECIDO,NEFROLITIASE,1,SIM,2,HOSPITALIZACAO,2,DESCONHECIDO,2021-02-03,NaT,0,DESCONHECIDO,2,CONCOMITANTE,METFORMINA,2,DESCONHECIDO,4,MEDICO,f2b5a4e2b8a3495b0addc13f1401bde6ced80317afb546...,2025-11-17,9999-12-31,2025-11-17,True
3,BR-ANVISA-300345102,4,25,SP,SAO PAULO,6,"EREPORTING - INDUSTRIA, CARGA E2B",5,EMPRESA FARMACEUTICA,BR-ANVISA-300345102,2025-07-21,2025-07-21,NaT,1,NOTIFICACAO ESPONTANEA,0,NAO INFORMADO,1974-11-23,46,ANO(S),0,DESCONHECIDO,DESCONHECIDO,0,DESCONHECIDO,...,DESCONHECIDO,DESCONHECIDO,CHOQUE ANAFILATICO,1,SIM,3,AMEACA R VIDA,2,DESCONHECIDO,2021-03-02,NaT,0,DESCONHECIDO,1,SUSPEITO,SORO ANTIBOTROPICO (PENTAVALENTE),2,DESCONHECIDO,3,FARMACEUTICO,cfb2b052c12cbf900a894b94a7573ea7323d28b3c99455...,2025-11-17,9999-12-31,2025-11-17,True
4,BR-ANVISA-300212385,5,25,SP,SAO PAULO,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,BR-ANVISA-300212385,2023-09-27,2023-09-27,NaT,1,NOTIFICACAO ESPONTANEA,0,NAO INFORMADO,NaT,0,DESCONHECIDO,0,DESCONHECIDO,DESCONHECIDO,0,DESCONHECIDO,...,DESCONHECIDO,DESCONHECIDO,ABSCESSO NASAL,1,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,2,DESCONHECIDO,NaT,NaT,0,DESCONHECIDO,1,SUSPEITO,VACINA ADSORVIDA COVID-19 (INATIVADA),2,DESCONHECIDO,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,0cd029bb9e8b797620238822ce1832f78036ec9b141c93...,2025-11-17,9999-12-31,2025-11-17,True


In [14]:
%%sql 
select * from fat_medicamentos
LIMIT 5

Running query in 'duckdb:///:memory:'

,id,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,DETENTOR_REGISTRO,CONCENTRACAO,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO,RELACAO_MEDICAMENTO_EVENTO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,COMPONENTE_SUSPEITO_VALOR,COMPONENTE_SUSPEITO_CHAVE,...,PROB_ADIC_SUPERDOSE,PROB_ADIC_USO_INCORRETO,PROB_ADIC_USO_OFF_LABEL_USO_SEM_REGISTRO,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,FORMA_FARMACEUTICA_SCORE,FORMA_FARMACEUTICA_FUZZY,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR,DETENTOR_REGISTRO_SCORE,DETENTOR_REGISTRO_FUZZY,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_CHAVE,VIA_ADMINISTRACAO_CHAVE_VALOR,VIA_ADMINISTRACAO_MAE_PAI_VALOR,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR
0,BR-ANVISA-300000004,BR-ANVISA-300000004,OXACILINA,OXACILLIN SODIUM,J01CF,TEUTO,None,NaN,None,None,250 MILLIGRAM (MG),6 HORA,None,4 DIA,2018-11-24,NaT,None,INFUSÃO INTRAVENOSA,None,5833018,2025-11-17,SUSPEITO,0,DESCONHECIDO,0,...,0,0,0,DIA,4,4.0,0.0,None,0,DESCONHECIDO,90.0,None,None,DESCONHECIDO,0,6,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
1,BR-ANVISA-300000005,BR-ANVISA-300000005,PARACEMOL,PARACETAMOL,N02BE,None,None,NaN,None,None,None,None,None,None,2018-11-22,2018-11-22,None,ORAL,None,None,2025-11-17,SUSPEITO,0,DESCONHECIDO,0,...,0,0,0,DESCONHECIDO,0,NaN,0.0,None,0,DESCONHECIDO,0.0,None,None,DESCONHECIDO,0,5,5,desconhecido,0,NaN,NaN,NaN,desconhecido,0,NaN
2,BR-ANVISA-300000007,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,C03,None,None,NaN,None,None,250 MILLIGRAM (MG),6 HORA,250MG A CADA 6 HORAS,None,2018-11-03,2018-11-15,None,ORAL,None,None,2025-11-17,SUSPEITO,0,DESCONHECIDO,0,...,0,0,0,DESCONHECIDO,0,NaN,0.0,None,0,DESCONHECIDO,0.0,None,None,DESCONHECIDO,0,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
3,BR-ANVISA-300000007,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,N03AX,None,None,NaN,None,None,250 MILLIGRAM (MG),6 HORA,250MG A CADA 6 HORAS,None,2018-11-03,2018-11-15,None,ORAL,None,None,2025-11-17,SUSPEITO,0,DESCONHECIDO,0,...,0,0,0,DESCONHECIDO,0,NaN,0.0,None,0,DESCONHECIDO,0.0,None,None,DESCONHECIDO,0,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
4,BR-ANVISA-300000007,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,S01EC,None,None,NaN,None,None,250 MILLIGRAM (MG),6 HORA,250MG A CADA 6 HORAS,None,2018-11-03,2018-11-15,None,ORAL,None,None,2025-11-17,SUSPEITO,0,DESCONHECIDO,0,...,0,0,0,DESCONHECIDO,0,NaN,0.0,None,0,DESCONHECIDO,0.0,None,None,DESCONHECIDO,0,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0


In [15]:
%%sql 
select * from dim_atc
where ATC_CODE_LEVEL_4 ='L04AB'

Running query in 'duckdb:///:memory:'

,ATC_CODE_LEVEL_1,ATC_CODE_LEVEL_2,ATC_CODE_LEVEL_3,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL,ATC_CODE_LEVEL_1_LEVEL_NAME,ATC_CODE_LEVEL_2_LEVEL_NAME,ATC_CODE_LEVEL_3_LEVEL_NAME,ATC_CODE_LEVEL_4_LEVEL_NAME,ATC_CODE_LEVEL_5_LEVEL_NAME,ATC_CODE_LEVEL,ATC_NAME_LEVEL,DDD_VALUE,UNIT,ADM_R,COMMENT
0,L,L04,L04A,L04AB,L04AB01,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,ETANERCEPT,L04AB01,etanercept,7.00,mg,P,None
1,L,L04,L04A,L04AB,L04AB02,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,INFLIXIMAB,L04AB02,infliximab,3.75,mg,P,None
2,L,L04,L04A,L04AB,L04AB03,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,AFELIMOMAB,L04AB03,afelimomab,NaN,None,None,None
3,L,L04,L04A,L04AB,L04AB04,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,ADALIMUMAB,L04AB04,adalimumab,2.90,mg,P,None
4,L,L04,L04A,L04AB,L04AB05,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,CERTOLIZUMAB PEGOL,L04AB05,certolizumab pegol,14.00,mg,P,None
5,L,L04,L04A,L04AB,L04AB06,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,GOLIMUMAB,L04AB06,golimumab,1.66,mg,P,None
6,L,L04,L04A,L04AB,L04AB07,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,OPINERCEPT,L04AB07,opinercept,7.00,mg,P,None
7,L,L04,L04A,L04AB,None,4,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,None,L04AB,Tumor necrosis factor alpha (TNF-alpha) inhibi...,NaN,None,None,None


In [16]:
%%sql 
select * from fat_reacoes
LIMIT 5

Running query in 'duckdb:///:memory:'

,id,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,ATUALIZADO,PK_SOC,PK_LLT,GRAVE_TIPO,GRAVE_VALOR,DESFECHO_TIPO,DESFECHO_VALOR,DURACAO_VALOR,DURACAO_TIPO,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,BR-ANVISA-300000004,NaT,NaT,3 dia,2025-11-17,16,4114,Não,2,Recuperado/Resolvido,3,3.0,DIA(S),0,0,0,0,0,0
1,BR-ANVISA-300000005,BR-ANVISA-300000005,2018-11-22,2018-11-22,desconhecido,2025-11-17,9,9107,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido,0,0,0,0,1,0
2,BR-ANVISA-300000007,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,2025-11-17,16,3903,Sim,1,Recuperado/Resolvido,3,2.0,DIA(S),0,0,0,0,1,0
3,BR-ANVISA-300000008,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,2025-11-17,12,11411,Sim,1,Recuperado/Resolvido,3,5.0,DIA(S),0,0,0,0,1,0
4,BR-ANVISA-300000010,BR-ANVISA-300000010,NaT,NaT,desconhecido,2025-11-17,8,1994,Sim,1,Recuperado/Resolvido,3,NaN,desconhecido,0,0,0,1,0,0


# 4. Como fazer join com a atc?

In [6]:
%%sql 

with fat_med as (
    select fat_medicamentos.*,
    (case 
    when atc.ATC_CODE_LEVEL_4 in ('L04AB', 'L01FA', 'L04AC','L04AF') 
        then concat(atc.ATC_CODE_LEVEL_4,'_',atc.ATC_CODE_LEVEL_4_LEVEL_NAME) 
        else 'Outros' 
        end) as ATC_CODE_LEVEL_4_LEVEL_NAME
    from fat_medicamentos
    inner join dim_atc as atc
    on fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 and 
    atc.ATC_CODE_LEVEL_5 is null
)

select
ATC_CODE_LEVEL_4_LEVEL_NAME
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med
group by 1
ORDER BY 1

Running query in 'duckdb:///:memory:'

,ATC_CODE_LEVEL_4_LEVEL_NAME,qt_notificacoes
0,L01FA_CD20 (CLUSTERS OF DIFFERENTIATION 20) IN...,3114
1,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,17261
2,L04AC_INTERLEUKIN INHIBITORS,5672
3,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,416
4,Outros,271583


# 5. Classificando atc_level_5 para estudo

In [17]:
%%sql
DROP TABLE IF EXISTS fat_med_w_atc_level_5;

CREATE TABLE fat_med_w_atc_level_5 AS
with fat_med as (
SELECT 
    fat_medicamentos.*, atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
    (
        CASE 
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L01FA', 'L04AC','L04AF') 
                THEN (
                    CASE 
                        /*Rituximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%' THEN 'L01FA01_Rituximab'
                        /*Abatacept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%' THEN 'L04AA24_Abatacept'
                        /*Etanercept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%' THEN 'L04AB01_Etanercept'
                        /*Infliximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%' THEN 'L04AB02_Infliximab'
                        /*Adalimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%' THEN 'L04AB04_Adalimumab'
                        /*Certolizumab Pegol*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumab Pegol'   
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumabe pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        
                        /*Golimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%' THEN 'L04AB06_Golimumab'
                        /*Tocilizumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumab'
                        /*Baricitinib */
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinib'
                        /*Upadacitinib*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%upadacitinib%' THEN 'L04AF03_Upadacitinib' 
                        ELSE 'missing'
                    END
                )
            ELSE 'Outros'
        END
    ) AS PRINCIPIOS_ATIVOS_CLEAN
FROM fat_medicamentos
INNER JOIN dim_atc AS atc
    ON fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 
    AND atc.ATC_LEVEL = 4
)
select 
    (case when PRINCIPIOS_ATIVOS_CLEAN in ('missing','Outros') then 'Outros' 
        else concat(ATC_CODE_LEVEL_4,'_',ATC_CODE_LEVEL_4_LEVEL_NAME) end) as ATC_LEVEL_4_NAME
,*
from fat_med

Running query in 'duckdb:///:memory:'

,Success


In [18]:
%%sql
select
ATC_LEVEL_4_NAME
,PRINCIPIOS_ATIVOS_CLEAN
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5
group by 1,2
ORDER BY 1,2

Running query in 'duckdb:///:memory:'

,ATC_LEVEL_4_NAME,PRINCIPIOS_ATIVOS_CLEAN,qt_notificacoes
0,L01FA_CD20 (CLUSTERS OF DIFFERENTIATION 20) IN...,L01FA01_Rituximab,2993
1,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB01_Etanercept,372
2,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB02_Infliximab,10529
3,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB04_Adalimumab,1616
4,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB05_Certolizumab Pegol,1221
5,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB06_Golimumab,3743
6,L04AC_INTERLEUKIN INHIBITORS,L04AC07_Tocilizumab,854
7,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,L04AF02_Baricitinib,94
8,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,L04AF03_Upadacitinib,267
9,Outros,Outros,271583


In [18]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AC' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc
limit 10

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,SECUKINUMAB,1487
1,USTEKINUMAB,982
2,USTEQUINUMABE,817
3,SECUQUINUMABE,552
4,RISANKIZUMAB,193
5,RISANQUIZUMABE,159
6,IXEQUIZUMABE,150
7,GUSELCUMABE,131
8,RISANKIZUMAB RZAA,115
9,GUSELKUMAB,107


In [19]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L01FA' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,OBINUTUZUMABE,75
1,OBINUTUZUMAB,46
2,OFATUMUMAB,5


In [20]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AB' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes


In [21]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AF' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,CITRATO DE TOFACITINIBE,37
1,TOFACITINIBE,18
2,RITLECITINIB TOSILATE,2
3,RUXOLITINIBE,1
4,DEUCRAVACITINIBE,1
